<a href="https://colab.research.google.com/github/Ellizence01/BSC_DPDM2025/blob/main/LSTM_V23_11_TMD_ALLcluster_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.1 MB/s eta 0:00:00


## Library Version 22_4 : Dropout between later, Dense graphs, USe Otograohic precitaiton edges caled from U, W CW, Dem and leedside windside, AttentionLSTM_V8, No Quantile Mapping, adaptive_huber_loss with percentile_loss to handle Extreme Events. Input includes calculated indices: PDO, ONI, SWM, DMI, and downloaded indices: MEIV2, BSISO, and MJO. , NEW NE index


In [ ]:
import os
import pandas as pd
import numpy as np
import sys
import datetime
import copy
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import pickle
import random
import matplotlib.pyplot as plt
import copy
from dateutil.relativedelta import relativedelta
import time
import inspect
import time as countdown
from tqdm import tqdm
import time as countdown
from torch_geometric.data import HeteroData




import torch
import torch.nn as nn
from torch.nn.parameter import Parameter
import torch.nn.functional as F
from torch.nn import Parameter
from torch import Tensor


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')



In [ ]:

import os, sys

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Hyperparameters

In [ ]:
algo='LSTM_V23_11'

num_epoch = 300
window_size = 24  #Window size
horizon = 12   #Prediction horizon
TBPTT_K = 24

learn_rate =0.001
weight_decay =1e-4
cluster =7 #choose cluster of the HII stations
k_fold_num=2
percentile_loss =0.95

# PARAMETERS FOR SMALL DATA
noise_level = 0.05  # Standard deviation of noise (5% of signal)

min_epochs = 20
patience = 100
LOG_EVERY = 20


hidden_size_list =[128]
num_layers_list =[2]
drop_out_list=[0.4]



## Load dataset

In [ ]:
X_index = pd.read_csv('/content/drive/MyDrive/Workshop Aj.Tohn/X_variable(Index)_operation_v4.csv')
Y_rainfall = pd.read_csv('/content/drive/MyDrive/Workshop Aj.Tohn/Y_variable(Rainfall).csv')
# Convert 'DATE' column in Y_df to datetime objects if it's not already
X_index['DATE'] = pd.to_datetime(X_index['DATE'])

# Format the dates in Y_df as 'YYYY-MM-01'
X_index['DATE'] = X_index['DATE'].dt.strftime('%Y-%m-01')



node = pd.read_csv('/content/drive/MyDrive/Workshop Aj.Tohn/Node_table_TMD.csv')


path_feature = os.path.join("/content/drive/MyDrive/Workshop Aj.Tohn/lookup_table_reanalysis_v5.csv")
feature = pd.read_csv(path_feature, header= "infer")



In [ ]:
Y_rainfall = Y_rainfall.rename(columns={'code': 'Node'})

Y_rainfall_clean = Y_rainfall.copy()
num_feature = len(feature[feature['Cluster']==cluster])

In [ ]:

def apply_time_lag(df, column_name, lag):
    # Create a new column name with the lag suffix
    new_column_name = f"{column_name}_lag{lag}"

    # Shift the column values by the specified lag
    df[new_column_name] = df[column_name].shift(lag)

    return df

# Define the time lags for each climate index

time_lags = {

    'MEIV2': [1, 2, 3],
    'RMM_AMPLITUDE': [1],
    'PHASE': [1],
    'PDO': [1, 2],
    'ONI': [1, 2],
    'DMI': [1, 2],
    'BSISO1': [1],
    'BSISO1-Phase': [1],
    'SWM': [1],

}




# Apply time lags to each specified column in df_merge_type
for column, lag in time_lags.items():
      if column in X_index.columns:
            for lag_item in lag:
                X_index = apply_time_lag(X_index, column, lag_item)
      else:
        print(f"Warning: Column '{column}' not found in df_merge_type. Skipping.")


Data cleansing

In [ ]:
HQ_date_get_month_df  =  Y_rainfall.copy()
Y_rainfall_clean =  Y_rainfall.copy()
HQ_date_get_month_df['DATE'] = pd.to_datetime(Y_rainfall['DATE']).dt.month

month_avg_list = []
for month in range(1, 12 + 1):
    # Filter the data for the current year
    HQ_data = HQ_date_get_month_df[HQ_date_get_month_df['DATE'] == month]

    # Average rainfall
    av_r = HQ_data.iloc[:, 1:].median()

    # Create a dictionary to hold the data for this year
    month_dict = {'MONTH': month}
    month_dict.update(av_r)

    # Append the data for this year to the list
    month_avg_list.append(month_dict)

HQ_month_avg = pd.DataFrame(month_avg_list)

for i in range(1, len(Y_rainfall_clean.columns)):
    for j in range(0, len(Y_rainfall_clean)):
        if(np.isnan(Y_rainfall_clean.iloc[j, i])):
             Y_rainfall_clean.iloc[j, i] = HQ_month_avg.iloc[pd.to_datetime(Y_rainfall_clean.iloc[:, 0]).dt.month[j]-1, i]

In [ ]:
Y_rainfall_clean['DATE'].max()

'2025-03-01'

In [ ]:
start_training_date = '1982-01-01'


X_index['DATE'] = pd.to_datetime(X_index['DATE'])
Y_rainfall_clean['DATE'] = pd.to_datetime(Y_rainfall_clean['DATE'])
max_date_X = X_index['DATE'].max()
max_date_Y = Y_rainfall_clean['DATE'].max()
# Minimum date possible

print("Maximum date possible : ",start_training_date)


# Maximum date possible
max_date_possible = min([max_date_X,max_date_Y])
print("Maximum date possible : ",max_date_possible)

#Make condition
con_date_X = (X_index['DATE'] >= start_training_date) & (X_index['DATE'] <= max_date_possible)
con_date_Y = (Y_rainfall_clean['DATE'] >= start_training_date) & (Y_rainfall_clean['DATE'] <= max_date_possible)

#Final select
X_index_interval_date = X_index.loc[con_date_X,:]
Y_rainfall_clean = Y_rainfall_clean.loc[con_date_Y,:]


Y_rainfall_interval_date = copy.deepcopy(Y_rainfall_clean)

Maximum date possible :  1982-01-01
Maximum date possible :  2025-03-01 00:00:00


In [ ]:
start_training_date = Y_rainfall_clean['DATE'].min()

X_index['DATE'] = pd.to_datetime(X_index['DATE'])
Y_rainfall_clean['DATE'] = pd.to_datetime(Y_rainfall_clean['DATE'])
max_date_X = X_index['DATE'].max()
max_date_Y = Y_rainfall_clean['DATE'].max()
# Minimum date possible

print("Maximum date possible : ",start_training_date)


# Maximum date possible
max_date_possible = min([max_date_X,max_date_Y])
print("Maximum date possible : ",max_date_possible)

#Make condition
con_date_X = (X_index['DATE'] >= start_training_date) & (X_index['DATE'] <= max_date_possible)
con_date_Y = (Y_rainfall_clean['DATE'] >= start_training_date) & (Y_rainfall_clean['DATE'] <= max_date_possible)

#Final select
X_index_interval_date = X_index.loc[con_date_X,:]
Y_rainfall_clean = Y_rainfall_clean.loc[con_date_Y,:]


Y_rainfall_interval_date = copy.deepcopy(Y_rainfall_clean)

Maximum date possible :  1982-01-01 00:00:00
Maximum date possible :  2025-03-01 00:00:00


In [ ]:
#Count null value of each indexs
print("Number of index columns which contain null value : ",len(X_index_interval_date.isnull().sum()[X_index_interval_date.isnull().sum()>0]), sep="")
print("Number of rainfall station which contain null value : ",len(Y_rainfall_interval_date.isnull().sum()[Y_rainfall_interval_date.isnull().sum()>0]), sep="")

Number of index columns which contain null value : 0
Number of rainfall station which contain null value : 0


## Convert DATE columns to index columns

In [ ]:
def convert_columns_to_index_columns(df,col_name):
    df.loc[:,col_name] = pd.to_datetime(df[col_name])
    df = df.set_index(col_name, inplace=False)
    return df

- independend variable dataframe

In [ ]:
X_index_df_ready = convert_columns_to_index_columns(X_index_interval_date, 'DATE')
Y_rainfall_df_ready = convert_columns_to_index_columns(Y_rainfall_interval_date, 'DATE')

In [ ]:
print("nrow of X : {}".format(X_index_df_ready.shape[0]))
print("nrow of Y : {}".format(X_index_df_ready.shape[0]))

nrow of X : 519
nrow of Y : 519


## Standardize

In [ ]:

#scaler_X = MinMaxScaler()
scaler_X = StandardScaler()

X_normalize_data = scaler_X.fit_transform(X_index_df_ready)
X_index_normalized_df = pd.DataFrame(X_normalize_data, columns=X_index_df_ready.columns)

scaler_Y = StandardScaler()

Y_normalize_data = scaler_Y.fit_transform(Y_rainfall_df_ready)
Y_rainfall_normalized_df = pd.DataFrame(Y_normalize_data, columns=Y_rainfall_df_ready.columns)

In [ ]:

# Save the scaler
with open('/content/drive/MyDrive/Workshop Aj.Tohn/scaler_Y.pkl', 'wb') as file:
    pickle.dump(scaler_Y, file)
# Save the scaler
with open('/content/drive/MyDrive/Workshop Aj.Tohn/scaler_X.pkl', 'wb') as file:
    pickle.dump(scaler_X, file)


# Load the scaler for inference
with open('/content/drive/MyDrive/Workshop Aj.Tohn/scaler_Y.pkl', 'rb') as file:
    scaler_Y = pickle.load(file)

## Summarises all climate index

In [ ]:
#Get date range from original files because after normalization index will reset
og_date_range = X_index_interval_date['DATE']
X_index_normalized_df.set_index(og_date_range, inplace=True)
Y_rainfall_normalized_df.set_index(og_date_range, inplace=True)

n_feature_climate = X_index_normalized_df.shape[1]
feature_climate = X_index_normalized_df.columns
print("Number of climate index feature : ",n_feature_climate)
print("All climate index : \n",feature_climate)

Number of climate index feature :  24
All climate index : 
 Index(['MEIV2', 'RMM_AMPLITUDE', 'PHASE', 'BSISO1', 'BSISO1-Phase', 'DMI',
       'ONI', 'PDO', 'SWM', 'NE', 'MEIV2_lag1', 'MEIV2_lag2', 'MEIV2_lag3',
       'RMM_AMPLITUDE_lag1', 'PHASE_lag1', 'PDO_lag1', 'PDO_lag2', 'ONI_lag1',
       'ONI_lag2', 'DMI_lag1', 'DMI_lag2', 'BSISO1_lag1', 'BSISO1-Phase_lag1',
       'SWM_lag1'],
      dtype='object')


## Add cos sin function to X

### Combine static feature

In [ ]:
X_index_normalized_df['Month_sin'] = np.sin(2 * np.pi * X_index_normalized_df.index.month / 12)
X_index_normalized_df['Month_cos'] = np.cos(2 * np.pi * X_index_normalized_df.index.month / 12)

In [ ]:
station_id_list = Y_rainfall_normalized_df.columns
list_table = []
for code_st in station_id_list:
  #print(code_st)

  #print(static_value)
  eindices_df = X_index_normalized_df.copy()



  eindices_df[code_st] = Y_rainfall_normalized_df[code_st]

  list_table.append(eindices_df)

In [ ]:
n_feature_climate = X_index_normalized_df.shape[1]
feature_climate = X_index_normalized_df.columns
print("Number of climate index feature : ",n_feature_climate)
print("All climate index : \n",feature_climate)

Number of climate index feature :  26
All climate index : 
 Index(['MEIV2', 'RMM_AMPLITUDE', 'PHASE', 'BSISO1', 'BSISO1-Phase', 'DMI',
       'ONI', 'PDO', 'SWM', 'NE', 'MEIV2_lag1', 'MEIV2_lag2', 'MEIV2_lag3',
       'RMM_AMPLITUDE_lag1', 'PHASE_lag1', 'PDO_lag1', 'PDO_lag2', 'ONI_lag1',
       'ONI_lag2', 'DMI_lag1', 'DMI_lag2', 'BSISO1_lag1', 'BSISO1-Phase_lag1',
       'SWM_lag1', 'Month_sin', 'Month_cos'],
      dtype='object')


## Split dataframe to add static feature

### Import meta data

### Add new date and latitude-longitude format columns

## Split training and testing

### Define start and end date of each interval

# Train Fold 1 = 2018-2020, Val = 2018-2020, Test = 2019 - 2021

# Train Fold 2 = 2018-2022, Val = 2021-2023, Test = 1/2022 - 12/2024

In [ ]:
#Train interval

start_interval_train1 =  pd.to_datetime(min(Y_rainfall_normalized_df.index))
#maximum_interval_train = max(Y_rainfall_normalized_df.index[0:-6])


end_interval_train1 = start_interval_train1 + relativedelta(months=37*12+11)


#Test1 interval
start_interval_val1 = start_interval_train1 + relativedelta(months=36*12)
end_interval_val1 =  start_interval_train1 + relativedelta(months=38*12+11)



#Test1 interval
start_interval_test1 = start_interval_train1 + relativedelta(months=37*12)
end_interval_test1 = start_interval_train1 + relativedelta(months=39*12+11)
#end_interval_test = start_interval_test + relativedelta(months=backward_months+17)
#Display
print("############################################################")
print("TRAINING Fold 1")
print('Start interval date of training set: ', start_interval_train1)
#print('Maximum interval date : ', maximum_interval_train)
print('End interval date of training set :', end_interval_train1)

print("############################################################")
print("VALIDATION Fold 1")
print('Start interval date of testing set: ', start_interval_val1)
print('End interval date of testing set :', end_interval_val1)
print("############################################################")



print("############################################################")
print("TESTING Fold 1")
print('Start interval date of testing set: ', start_interval_test1)
print('End interval date of testing set :', end_interval_test1)
print("############################################################")
#if cluster in [4 ,5,6,7, 8,9 ,10,11,12]:
#    start_interval_train1 = start_interval_val1 +  relativedelta(months=1)

#    end_interval_train2 = start_interval_train1 +  relativedelta(months=4*12+12)


    #Train interval
#    start_interval_val2 = start_interval_train1 +  relativedelta(months=3*12-1)
#    end_interval_val2 = start_interval_train1 +  relativedelta(months=5*12+10)



    #Test2 interval
#    start_interval_test2 = start_interval_train1 +  relativedelta(months=3*12+9)
#    end_interval_test2 = start_interval_train1 +  relativedelta(months=6*12+11-3)

#else:
start_interval_train1 = start_interval_train1

end_interval_train2 = start_interval_train1 + relativedelta(months=40*12)


    #Train interval
start_interval_val2 = start_interval_train1  + relativedelta(months=39*12)
end_interval_val2 = start_interval_train1 + relativedelta(months=41*12+11)



    #Test2 interval
start_interval_test2 = start_interval_train1 + relativedelta(months=40*12)
end_interval_test2 = start_interval_train1 + relativedelta(months=42*12+11)


#Display
print("############################################################")
print("TRAINING Fold 2")
print('Start interval date of training set: ', start_interval_train1)
#print('Maximum interval date : ', maximum_interval_train)
print('End interval date of training set :', end_interval_train2)

print("############################################################")
print("VALIDATION Fold 2")
print('Start interval date of testing set: ', start_interval_val2)
print('End interval date of testing set :', end_interval_val2)
print("############################################################")



print("############################################################")
print("TESTING Fold 2")
print('Start interval date of testing set: ', start_interval_test2)
print('End interval date of testing set :', end_interval_test2)
print("############################################################")

# Train Fold 1 = 1982-2019, Val = 2018-2020, Test = 2019 - 2021

# Train Fold 2 = 1982-2022, Val = 2021-2023, Test = 2022 - 2024

############################################################
TRAINING Fold 1
Start interval date of training set:  1982-01-01 00:00:00
End interval date of training set : 2019-12-01 00:00:00
############################################################
VALIDATION Fold 1
Start interval date of testing set:  2018-01-01 00:00:00
End interval date of testing set : 2020-12-01 00:00:00
############################################################
############################################################
TESTING Fold 1
Start interval date of testing set:  2019-01-01 00:00:00
End interval date of testing set : 2021-12-01 00:00:00
############################################################
############################################################
TRAINING Fold 2
Start interval date of training set:  1982-01-01 00:00:00
End interval date of training set : 2022-01-01 00:00:00
############################################################
VALIDATION Fold 2
Start interval date of testing set:  2

### Split training and testing

In [ ]:
#Split test and train

Train_df_list1, Val_df_list1, Test_df_list1,Train_df_list2,  Val_df_list2 ,Test_df_list2 = [], [],[],[],[],[]
for i_table in range(len(list_table)):
  Train_df_list1.append(list_table[i_table][start_interval_train1:end_interval_train1])
  Val_df_list1.append(list_table[i_table][start_interval_val1:end_interval_val1])
  Test_df_list1.append(list_table[i_table][start_interval_test1:end_interval_test1])
  Train_df_list2.append(list_table[i_table][start_interval_train1:end_interval_train2])
  Val_df_list2.append(list_table[i_table][start_interval_val2:end_interval_val2])
  Test_df_list2.append(list_table[i_table][start_interval_test2:end_interval_test2])

fold =[]
fold.append([Train_df_list1,Test_df_list1,Val_df_list1])
fold.append([Train_df_list2,Test_df_list2,Val_df_list2])


# Feature selection part

## Focus only feature of climate

In [ ]:
print("The first feature size that is climate index:", n_feature_climate)

The first feature size that is climate index: 26


In [ ]:
selected_index =[]
feature_cluster =  feature[feature['Cluster']==cluster]
for feature_index in range(len(feature_cluster)):
    for i in range(len(feature_climate)):
        if((feature_climate[i] == feature_cluster.iloc[feature_index,1]) and feature_cluster.iloc[feature_index,5] == 0):
                selected_index.append(i)

        elif(feature_cluster.iloc[feature_index,5] != 0):
            for i_lag in range(len(feature_climate)):
                if(feature_climate[i_lag] == feature_cluster.iloc[feature_index,1]+'_lag'+str(feature_cluster.iloc[feature_index,5])):
                    selected_index.append(i_lag)
                    break
                else:
                    continue  # Continue if the inner loop wasn't broken.
            break  # Inner loop was broken, break the outer.

selected_index.sort()
feature_climate[selected_index]


Index(['RMM_AMPLITUDE', 'BSISO1', 'DMI', 'SWM', 'NE', 'Month_sin',
       'Month_cos'],
      dtype='object')

## Feature selection
- Create binary position (length = climate feature)

In [ ]:
random.seed(10)
position_optimal = [0 for i in range(n_feature_climate)]
#position_remain = (Train_df_list[0].shape[1] - n_feature_climate)*[1]
position_remain = [0, 0, 1] # select rain

for id in selected_index:
  position_optimal[id]=1

position_all = position_optimal + position_remain
position_all_index = [index for index,value in enumerate(position_all) if value == 1]
climate_name_list = [feature_climate[index] for index,value in enumerate(position_optimal) if value == 1]

print("position_all_index : ", position_all_index)
print("Climate index : ", climate_name_list)

position_all_index :  [1, 3, 5, 8, 9, 24, 25, 28]
Climate index :  ['RMM_AMPLITUDE', 'BSISO1', 'DMI', 'SWM', 'NE', 'Month_sin', 'Month_cos']


## Select columns by position vector (position_all)

In [ ]:
random.seed(10)
position_optimal = [0 for i in range(n_feature_climate)]
#position_remain = (Train_df_list[0].shape[1] - n_feature_climate)*[1]
position_remain = [1] # select rain

for id in selected_index:
  position_optimal[id]=1

position_all = position_optimal + position_remain
position_all_index = [index for index,value in enumerate(position_all) if value == 1]
climate_name_list = [feature_climate[index] for index,value in enumerate(position_optimal) if value == 1]

print("position_all_index : ", position_all_index)
print("Climate index : ", climate_name_list)

position_all_index :  [1, 3, 5, 8, 9, 24, 25, 26]
Climate index :  ['RMM_AMPLITUDE', 'BSISO1', 'DMI', 'SWM', 'NE', 'Month_sin', 'Month_cos']


In [ ]:
def select_feature_f(list_df,position_feature_index):
    for i in range(len(list_df)):
        list_df[i] = list_df[i].iloc[:,position_feature_index]
    return list_df

In [ ]:
#Fold 1 = Fold[0] , Fold 2 = Fold[1]
#Test Fold 1 = Fold[0][0] , Train Fold 1 = Fold[0][1], Validation Fold 1 = Fold[0][2]
fold[0][0] = select_feature_f(list_df = fold[0][0], position_feature_index = position_all_index)
fold[0][1] = select_feature_f(list_df = fold[0][1], position_feature_index = position_all_index)
fold[0][2] = select_feature_f(list_df = fold[0][2], position_feature_index = position_all_index)
fold[1][0] = select_feature_f(list_df = fold[1][0], position_feature_index = position_all_index)
fold[1][1] = select_feature_f(list_df = fold[1][1], position_feature_index = position_all_index)
fold[1][2] = select_feature_f(list_df = fold[1][2], position_feature_index = position_all_index)

In [ ]:
len(Y_rainfall_normalized_df.columns)

132

## Read node location

In [ ]:
cluster_station= []
for index in range(0,len(Y_rainfall_normalized_df.columns)):
    if (node['quality'].iloc[index]==1 and node['cluster'].iloc[index]==cluster) :
            cluster_station.append(index)


In [ ]:
for index in range(len(cluster_station)):
    print(node['Node'].iloc[cluster_station[index]])

300201
300202
303201
303301
327501
328201
331201
331301
331401


Create node

Creat training test set for each station

Create static features

In [ ]:
num_feature = len(feature[feature['Cluster']==cluster])

## Check edges

In [ ]:

def extract_features(feature,j, window_size,horizon,data_list):
    xs_list= []
    for station in cluster_station:
        xs_new= np.array(data_list[station].iloc[:,feature][j:j+window_size])
        xs_list.append([xs_new])
    xs = np.array(xs_list)
    return xs

In [ ]:
climate_name_list

['RMM_AMPLITUDE', 'BSISO1', 'DMI', 'SWM', 'NE', 'Month_sin', 'Month_cos']

Add snapshot signal in = Windows , out = 12

Train_df_listCreate edges

In [ ]:
num_feature

7

Convert to homogenious structure

In [ ]:


# Station id
s=0



#Sliding window Train
fold_tensor = copy.deepcopy(fold)

for fold_index in range(k_fold_num):
    train_set =[]
    train= HeteroData()

    test_set =[]
    test= HeteroData()

    val_set =[]
    val = HeteroData()


    for j in range(len(fold[fold_index][0][s]) - window_size - horizon + 1):
        # if j%(window_size-1)==0: #
        #if j%3==0: #
        tan_x=[]
        rain_fall_x=[]
        rain_fall_y=[]

        for station in cluster_station:
            tan_x.append([(fold[fold_index][0][station][fold[fold_index][0][station].columns[-2]][j:j+window_size])])
            rain_fall_x.append([(fold[fold_index][0][station][fold[fold_index][0][station].columns[-1]][j:j+window_size])])
            rain_fall_y.append([(fold[fold_index][0][station][fold[fold_index][0][station].columns[-1]][j+window_size:j+window_size+horizon])])


        for climate_index in range(len(climate_name_list)):

            train[climate_name_list[climate_index]].x = torch.tensor(extract_features(climate_index,j, window_size,horizon,fold[fold_index][0]), dtype=torch.float32)
        train['station'].x = torch.tensor(np.array(rain_fall_x), dtype=torch.float32)
        train['station'].y = torch.tensor(np.array(rain_fall_y), dtype=torch.float32)

        train_set.append(copy.deepcopy(train))

    fold_tensor[fold_index][0] = train_set
    #Sliding window Test

    for j in range(len(fold[fold_index][1][s]) - window_size - horizon + 1):
        tan_x =[]
        rain_fall_x=[]
        rain_fall_y=[]
        for station in cluster_station:
            tan_x.append([(fold[fold_index][1][station][fold[fold_index][1][station].columns[-2]][j:j+window_size])])
            rain_fall_x.append([(fold[fold_index][1][station][fold[fold_index][1][station].columns[-1]][j:j+window_size])])
            rain_fall_y.append([(fold[fold_index][1][station][fold[fold_index][1][station].columns[-1]][j+window_size:j+window_size+horizon])])


        for climate_index in range(len(climate_name_list)):

            test[climate_name_list[climate_index]].x = torch.tensor(extract_features(climate_index,j, window_size,horizon,fold[fold_index][1]), dtype=torch.float32)


        test['station'].x = torch.tensor(np.array(rain_fall_x), dtype=torch.float32)
        test['station'].y = torch.tensor(np.array(rain_fall_y), dtype=torch.float32)


        test_set.append(copy.deepcopy(test))

    fold_tensor[fold_index][1] = test_set


    for j in range(len(fold[fold_index][2][s]) - window_size - horizon + 1):
        tan_x =[]
        rain_fall_x=[]
        rain_fall_y=[]
        for station in cluster_station:
            tan_x.append([(fold[fold_index][2][station][fold[fold_index][2][station].columns[-2]][j:j+window_size])])
            rain_fall_x.append([(fold[fold_index][2][station][fold[fold_index][2][station].columns[-1]][j:j+window_size])])
            rain_fall_y.append([(fold[fold_index][2][station][fold[fold_index][2][station].columns[-1]][j+window_size:j+window_size+horizon])])

        for climate_index in range(len(climate_name_list)):
            val[climate_name_list[climate_index]].x = torch.tensor(extract_features(climate_index,j, window_size,horizon,fold[fold_index][2]), dtype=torch.float32)

        val['station'].x = torch.tensor(np.array(rain_fall_x), dtype=torch.float32)
        val['station'].y = torch.tensor(np.array(rain_fall_y), dtype=torch.float32)



        val_set.append(copy.deepcopy(val))

    fold_tensor[fold_index][2] = val_set



In [ ]:
def extract_x(data):
    datax =[]
    i=0
    cluster_i=0
    for feature_index in range((num_feature)):

        datax.append([data.x[i][0].detach().cpu().numpy()])
        i=i+len(cluster_station)
    #result = torch.cat(datax, dim=1)
    for cluster_index in range(len(cluster_station)):

        datax.append([data.x[i+cluster_i][0].detach().cpu().numpy()])
        cluster_i=cluster_i+1
    return datax


In [ ]:

def accuracy2D_classic(y_pred, y_target):
    y_pred=y_pred.detach().cpu().numpy()
    y_target=y_target.detach().cpu().numpy()
    y_pred =np.reshape(y_pred,y_pred.shape[0]*y_pred.shape[1])
    y_target =np.reshape(y_target,y_target.shape[0]*y_target.shape[1])
    a = [1-min(1,abs((j-i)/i)) for j,i in zip(y_pred, y_target) if i !=0]
    for i in range(len(y_target)):
        if y_target[i]==0:
            a[i] = 1-min(1,2*abs(y_pred[i]-y_target[i])/abs(y_pred[i]+y_target[i]))

    return np.array(a)

In [ ]:
def transformback_future(pred,station,n):

    preds_temp = np.ones(Y_normalize_data.shape)
    for i in range(len(pred)):
        preds_temp[Y_normalize_data.shape[0]-n+i,station] =pred[pred.shape[0]-n+i]
    preds_values = scaler_Y.inverse_transform(preds_temp)
    preds_value = preds_values[preds_values.shape[0]-n:preds_values.shape[0],station]
    return (preds_value)

In [ ]:
import torch
from typing import Dict, Union, List

def nse_per_station(observations, predictions, station_indices=None):
    """
    Compute the Nash-Sutcliffe Efficiency (NSE) for each station separately.

    Args:
        observations (torch.Tensor): Ground truth values, shape (num_stations, num_timesteps).
        predictions (torch.Tensor): Predicted values, same shape as `observations`.
        station_indices (Dict[str, int], optional): Dictionary mapping station names to their
                                                  indices in the tensors.

    Returns:
        Dict[str, float]: Nash-Sutcliffe Efficiency (NSE) score for each station.
    """
    # Ensure inputs have the correct shape
    if observations.dim() != 2 or predictions.dim() != 2:
        raise ValueError("Inputs must be 2D tensors with shape (num_stations, num_timesteps)")

    if observations.shape != predictions.shape:
        raise ValueError(f"Shape mismatch: observations {observations.shape}, predictions {predictions.shape}")

    num_stations = observations.shape[0]

    # Initialize results dictionary
    if station_indices is None:
        # If no station names provided, use numerical indices
        station_indices = {str(i): i for i in range(num_stations)}

    nse_scores = {}

    # Calculate NSE for each station
    for station_name, idx in station_indices.items():
        # Get data for this station
        station_obs = observations[idx]
        station_pred = predictions[idx]

        # Skip stations with NaN values if needed
        if torch.isnan(station_obs).any() or torch.isnan(station_pred).any():
            # Handle NaN values by filtering them out
            valid_mask = ~(torch.isnan(station_obs) | torch.isnan(station_pred))
            station_obs = station_obs[valid_mask]
            station_pred = station_pred[valid_mask]

            # Skip if no valid data points remain
            if len(station_obs) == 0:
                nse_scores[station_name] = float('nan')
                continue

        # Calculate mean of observations for THIS STATION only
        mean_obs = torch.mean(station_obs)

        # Compute numerator (sum of squared residuals)
        numerator = torch.sum((station_obs - station_pred) ** 2)

        # Compute denominator (sum of squared deviations from mean of observations)
        denominator = torch.sum((station_obs - mean_obs) ** 2)

        # Avoid division by zero
        if denominator == 0:
            # If predictions match observations exactly when they're all the same value
            if numerator == 0:
                nse_scores[station_name] = 1.0
            else:
                nse_scores[station_name] = float('nan')  # NSE undefined
        else:
            # Compute NSE for this station
            station_nse = 1 - (numerator / denominator)
            nse_scores[station_name] = station_nse.item()

    return nse_scores



In [ ]:
def adaptive_huber_loss(y_pred, y_true, delta=1.0):
    """
    Huber loss with adaptive delta parameter based on data distribution

    Args:
        y_pred: Model predictions
        y_true: Ground truth values
        delta: Threshold parameter that determines quadratic vs. linear loss regions
    """
    # Calculate absolute error
    abs_error = torch.abs(y_pred - y_true)

    # Adapt delta based on data distribution (e.g., use 90th percentile)
    if delta is None:
        delta = torch.quantile(abs_error.detach(), 0.9)

    # Apply Huber loss formula
    quadratic_mask = (abs_error <= delta)
    linear_mask = ~quadratic_mask

    quadratic_loss = 0.5 * (abs_error ** 2)
    linear_loss = delta * (abs_error - 0.5 * delta)

    return torch.mean(quadratic_mask * quadratic_loss + linear_mask * linear_loss)

In [ ]:
def nse(observations, predictions):
    """
    Compute the Nash-Sutcliffe Efficiency (NSE) between observed and predicted values.

    Args:
        observations (torch.Tensor): Ground truth values, shape (batch_size, ...).
        predictions (torch.Tensor): Predicted values, same shape as `observations`.

    Returns:
        float: Nash-Sutcliffe Efficiency (NSE) score.
    """


    # Calculate the mean of observed values
    mean_obs = np.mean(observations)

    # Compute numerator (sum of squared residuals)
    numerator = np.sum((observations - predictions) ** 2)

    # Compute denominator (sum of squared deviations from mean of observations)
    denominator = np.sum((observations - mean_obs) ** 2)

    # Avoid division by zero
    if denominator == 0:
        return float('nan')  # NSE cannot be computed if observations have zero variance

    # Compute and return NSE
    nse_score = 1 - (numerator / denominator)
    return nse_score.item()  # Convert to Python float for readability


In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

def inverse_transform_station_with_nans(scaled_predictions, station_index, scaler_object):
    """
    Efficiently transforms scaled predictions for a single station that may contain NaN
    values, using a multi-feature scaler. NaNs in the input are preserved as NaNs
    in the output.

    Args:
        scaled_predictions (np.ndarray):
            A 1D NumPy array containing the scaled predictions (and potentially NaNs)
            for a single station.

        station_index (int):
            The column index of the station that these predictions belong to.

        scaler_object (object):
            The multi-feature scikit-learn scaler object (e.g., MinMaxScaler) that
            was fitted on the complete, original dataset.

    Returns:
        np.ndarray:
            A 1D NumPy array of the same size as the input, with predictions in their
            original scale and NaNs preserved in their original positions.
    """
    # --- 1. Identify the locations of valid data and NaNs ---
    # Create a boolean mask to find where the valid numbers are.
    valid_mask = ~np.isnan(scaled_predictions)

    # Extract only the valid, non-NaN scaled prediction values.
    valid_scaled_preds = scaled_predictions[valid_mask]

    # If there are no valid values to transform, return the original array.
    if valid_scaled_preds.size == 0:
        return scaled_predictions

    # --- 2. Use the dummy array trick ONLY for the valid data ---
    # Get the total number of features (stations) the scaler was trained on.
    n_features = scaler_object.n_features_in_

    # Create a small dummy array. Its height is the number of *valid* predictions.
    # Its width is the total number of features.
    dummy_array = np.zeros((len(valid_scaled_preds), n_features))

    # Inject the valid predictions into the correct station's column.
    dummy_array[:, station_index] = valid_scaled_preds

    # --- 3. Perform the inverse transformation ---
    # This is efficient as it only transforms the valid data points.
    unscaled_dummy_array = scaler_object.inverse_transform(dummy_array)

    # Extract the now unscaled *valid* predictions from the correct column.
    unscaled_valid_preds = unscaled_dummy_array[:, station_index]

    # --- 4. Re-insert the transformed values back into a full-sized array ---
    # Create a new array of the original size, filled with NaNs.
    final_predictions = np.full_like(scaled_predictions, np.nan, dtype=float)

    # Use the boolean mask from step 1 to place the unscaled valid values
    # back into their original positions.
    final_predictions[valid_mask] = unscaled_valid_preds

    return final_predictions

In [ ]:
def quantile_mapper(value, historical_preds, historical_obs):
        """
        Maps a value from the prediction distribution to the observation distribution.

        Args:
            value (float): The predicted value to be corrected (in mm).
            historical_preds (np.array): Sorted array of historical predictions.
            historical_obs (np.array): Sorted array of historical observations.

        Returns:
            float: The corrected value in mm.
        """
        # Find the percentile of the predicted value
        percentile = percentileofscore(historical_preds, value)

        # Find the corresponding value at that percentile in observations
        corrected_value = np.percentile(historical_obs, percentile, method='closest_observation')

        return corrected_value

In [ ]:
len(cluster_station)

9

# Geometric Temporal Torch Model

Which model to choose depends on which time-series task you work on.

- AttentionGCN-GRU is an enhancement of GCN-LTSM that uses attention
- The spatial aggregation uses GCN, the temporal aggregation a GRU
- We can pass in periods to get an embedding for several timesteps
- This embedding can be used to predict several steps into the future = output dimension

Model Attention : An implementation of the Attention Graph Convolutional Cell. For details see this paper: “GRAPH ATTENTION NETWORKS” https://arxiv.org/pdf/1710.10903


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

class EarlyStopping:
    """
    Early stopping handler for PyTorch training with loss history plotting.
    - Saves best model whenever val_metric improves AFTER min_epochs.
    - Stops only after `min_epochs` and patience exceeded.
    """
    def __init__(
        self,
        patience=7,
        min_delta=0.0,
        mode='min',
        save_path=None,
        verbose=False,
        min_epochs=min_epochs,     # ✅ do not track best or early-stop before this epoch
    ):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.mode = str(mode).lower()
        assert self.mode in ("min", "max"), "mode must be 'min' or 'max'"
        self.save_path = save_path
        self.verbose = bool(verbose)
        self.min_epochs = int(min_epochs)

        self.counter = 0
        self.best_score = None
        self.early_stop = False

        self.val_best = np.inf if self.mode == 'min' else -np.Inf

        # history
        self.val_loss_history = []
        self.train_loss_history = []
        self.epochs_history = []

    def _is_improvement(self, val_metric: float) -> bool:
        if self.best_score is None:
            return True

        if self.mode == 'min':
            return val_metric < (self.val_best - self.min_delta)
        else:
            return val_metric > (self.val_best + self.min_delta)

    def __call__(self, val_metric, train_metric, model, epoch):
        val_metric = float(val_metric)
        train_metric = float(train_metric)
        epoch = int(epoch)

        self.val_loss_history.append(val_metric)
        self.train_loss_history.append(train_metric)
        self.epochs_history.append(epoch)

        # ✅ FIXED: Completely ignore checkpointing & stopping logic before min_epochs
        if epoch < self.min_epochs:
            return False

        improved = self._is_improvement(val_metric)

        if improved:
            self.val_best = val_metric
            self.best_score = -val_metric if self.mode == 'min' else val_metric

            self.save_checkpoint(val_metric, model, epoch)

            self.counter = 0
        else:
            if self.best_score is not None:
                self.counter += 1
                if self.verbose:
                    print(f'EarlyStopping counter: {self.counter} out of {self.patience}')

        if self.counter >= self.patience:
            self.early_stop = True

        return self.early_stop

    def plot_loss_history(self, save_plot=True):
        plt.figure(figsize=(10, 6))
        plt.plot(self.epochs_history, self.train_loss_history, label='Training Loss', marker='o')
        plt.plot(self.epochs_history, self.val_loss_history, label='Validation Loss', marker='s')

        # ✅ FIXED: Only mark the "Best Val" from epochs >= min_epochs
        if len(self.val_loss_history) > 0:
            offset = min(self.min_epochs, len(self.val_loss_history) - 1)
            valid_hist = self.val_loss_history[offset:]

            if len(valid_hist) > 0:
                if self.mode == "min":
                    best_epoch_idx = offset + int(np.argmin(valid_hist))
                else:
                    best_epoch_idx = offset + int(np.argmax(valid_hist))

                best_val = float(self.val_loss_history[best_epoch_idx])
                best_epoch = self.epochs_history[best_epoch_idx]

                plt.axvline(x=best_epoch, color='r', linestyle='--', alpha=0.5)

                plt.annotate(
                    f'Best Val: {best_val:.4f}',
                    xy=(best_epoch, best_val),
                    xytext=(10, 10),
                    textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.5),
                    arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0')
                )

        plt.title('Training and Validation Loss Over Time')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()

        if save_plot and self.save_path is not None:
            plt.savefig(self.save_path + '_loss_history.png')

    def save_checkpoint(self, val_metric, model, epoch):
        """Saves model when validation metric improves."""
        if self.save_path is None:
            return

        if self.verbose:
            if self.mode == 'min':
                print(f'Validation metric decreased ({self.val_best:.6f} -> {val_metric:.6f}). Saving model @ epoch {epoch}...')
            else:
                print(f'Validation metric increased ({self.val_best:.6f} -> {val_metric:.6f}). Saving model @ epoch {epoch}...')

        checkpoint = {
            'epoch': int(epoch),
            'model_state_dict': model.state_dict(),
            'validation_metric': float(val_metric),
            'val_loss_history': self.val_loss_history,
            'train_loss_history': self.train_loss_history,
            'epochs_history': self.epochs_history
        }

        torch.save(checkpoint, self.save_path + '.pth')

## best training accuracy # AttentionLSTM_V7_NS (returns output, H_next, C_next)
# - Dropout between-layer only => NO feature-dropout inside this block
# - Also sets GATConv dropout to 0.0 to be STRICTLY between-layer only

## AttentionLSTM_V7_NS with Variational Dropout & Noise Injection Support

In [ ]:
import torch
import torch.nn as nn
from torch.nn.parameter import Parameter
import numpy as np
import pandas as pd
from tqdm import tqdm
import time as countdown
import os # Import os module

# -------------------------
# TemporalLSTM_V7_NS (Pure LSTM)
# - Removed GATv2Conv (Graph Attention)
# - Replaced with a Linear projection layer
# - Kept detrending and custom non-stationary gates
# -------------------------
class TemporalLSTM_V7_NS(nn.Module):
    def __init__(
        self,
        num_of_nodes: int,
        in_channels: int,
        out_channels: int,
        bias: bool = True,
        drop_out: float = 0.0,
        cond_dim: int = 0,
        cond_hidden: int = 64,
        level_hidden: int = 64,
        gatebias_hidden: int = 64,
        level_skip: float = 0.1,
        gatebias_scale: float = 0.5,
        station_emb_dim: int = 4,
        z_drop_out: float = 0.2,
        rec_drop_out: float = 0.2,
    ):
        super().__init__()
        self.num_of_nodes  = int(num_of_nodes)
        self.in_channels   = int(in_channels)
        self.out_channels  = int(out_channels)
        self.drop_out      = float(drop_out)
        self.z_drop_out    = float(drop_out if z_drop_out  is None else z_drop_out)
        self.rec_drop_out  = float(drop_out if rec_drop_out is None else rec_drop_out)
        self.level_skip    = float(level_skip)
        self.gatebias_scale = float(gatebias_scale)

        self.embed_dim = self.out_channels

        self.station_id_emb = nn.Embedding(self.num_of_nodes, station_emb_dim)
        self.total_dim      = self.embed_dim + station_emb_dim

        # ✅ REPLACED GAT WITH LINEAR PROJECTION
        self.input_proj = nn.Linear(self.in_channels, self.embed_dim, bias=bias)

        self.layer_norm = nn.LayerNorm(self.embed_dim)
        self.z_dropout  = nn.Dropout(self.z_drop_out)

        self.cond_dim = int(cond_dim)
        if self.cond_dim > 0:
            self.cond_mlp = nn.Sequential(
                nn.Linear(self.cond_dim, cond_hidden),
                nn.SiLU(),
                nn.Linear(cond_hidden, self.embed_dim),
            )

        self.level_net   = nn.Sequential(
            nn.Linear(self.total_dim, level_hidden),
            nn.SiLU(),
            nn.Linear(level_hidden, self.total_dim),
        )
        self.detrend_ln = nn.LayerNorm(self.total_dim)

        self.lstm_input_ln  = nn.LayerNorm(self.total_dim)
        self.lstm_hidden_ln = nn.LayerNorm(self.total_dim)

        gb_in_dim = self.total_dim + (self.embed_dim if self.cond_dim > 0 else 0)
        self.gatebias_net = nn.Sequential(
            nn.Linear(gb_in_dim, gatebias_hidden),
            nn.SiLU(),
            nn.Linear(gatebias_hidden, 4 * self.total_dim),
        )

        # LSTM weight matrices
        self.W_i = Parameter(torch.empty(self.total_dim, self.total_dim))
        self.U_i = Parameter(torch.empty(self.total_dim, self.total_dim))
        self.b_i = Parameter(torch.zeros(1, self.total_dim))

        self.W_f = Parameter(torch.empty(self.total_dim, self.total_dim))
        self.U_f = Parameter(torch.empty(self.total_dim, self.total_dim))
        self.b_f = Parameter(torch.zeros(1, self.total_dim))

        self.W_c = Parameter(torch.empty(self.total_dim, self.total_dim))
        self.U_c = Parameter(torch.empty(self.total_dim, self.total_dim))
        self.b_c = Parameter(torch.zeros(1, self.total_dim))

        self.W_o = Parameter(torch.empty(self.total_dim, self.total_dim))
        self.U_o = Parameter(torch.empty(self.total_dim, self.total_dim))
        self.b_o = Parameter(torch.zeros(1, self.total_dim))

        self.output_proj = nn.Linear(self.total_dim, self.embed_dim)
        self.reset_parameters()

    def reset_parameters(self):
        for p in [self.W_i, self.U_i, self.W_f, self.U_f,
                  self.W_c, self.U_c, self.W_o, self.U_o]:
            nn.init.xavier_uniform_(p)
        nn.init.constant_(self.b_f, 0.8)

    def forward(self, X, cond=None, H=None, C=None):
        # --- Pure Temporal encoding ---
        Z = self.input_proj(X)   # [N, embed_dim]
        Z = self.layer_norm(Z)
        if self.z_drop_out > 0:
            Z = self.z_dropout(Z)

        if X.size(0) != self.num_of_nodes:
            raise ValueError(f"X has {X.size(0)} nodes but station_id_emb expects num_of_nodes={self.num_of_nodes}.")

        node_ids = torch.arange(self.num_of_nodes, device=X.device)
        identity  = self.station_id_emb(node_ids)
        Z_combined = torch.cat([Z, identity], dim=-1)

        if H is None:
            H = torch.zeros(Z_combined.size(0), self.total_dim, device=Z.device, dtype=Z.dtype)
        if C is None:
            C = torch.zeros(Z_combined.size(0), self.total_dim, device=Z.device, dtype=Z.dtype)

        level = self.level_net(Z_combined)
        Zd    = self.detrend_ln(Z_combined - level)

        if (self.cond_dim > 0) and (cond is not None):
            if cond.dim() == 1:
                cond = cond.unsqueeze(0)
            cvec   = self.cond_mlp(cond.to(Z.device).to(Z.dtype)).expand(Z.size(0), -1)
            gb_in  = torch.cat([level, cvec], dim=-1)
        else:
            gb_in = level

        gate_bias      = self.gatebias_net(gb_in)
        bi, bf, bc, bo = gate_bias.chunk(4, dim=-1)

        Zd_norm = self.lstm_input_ln(Zd)
        H_norm  = self.lstm_hidden_ln(H)

        Zd_used = torch.nn.functional.dropout(Zd_norm, p=self.drop_out,     training=self.training)
        H_used  = torch.nn.functional.dropout(H_norm,  p=self.rec_drop_out, training=self.training)

        s = self.gatebias_scale
        i_gate = torch.sigmoid(Zd_used @ self.W_i + H_used @ self.U_i + self.b_i + s * torch.tanh(bi))
        f_gate = torch.sigmoid(Zd_used @ self.W_f + H_used @ self.U_f + self.b_f + s * torch.tanh(bf))
        g_gate = torch.tanh(   Zd_used @ self.W_c + H_used @ self.U_c + self.b_c + s * torch.tanh(bc))
        o_gate = torch.sigmoid(Zd_used @ self.W_o + H_used @ self.U_o + self.b_o + s * torch.tanh(bo))

        C_next = f_gate * C + i_gate * g_gate
        H_next = o_gate * torch.tanh(C_next)

        if self.level_skip > 0:
            H_next = H_next + self.level_skip * level

        output = self.output_proj(H_next)
        return output, H_next, C_next


# -------------------------
# PureLSTMModel
# - Removed DropEdge
# - Removed edge dependencies
# -------------------------
class PureLSTMModel(nn.Module):
    def __init__(
        self,
        num_of_nodes,
        input_size,
        hidden_size,
        num_layers,
        drop_out,
        output_size,
        z_drop_out: float = None,
        rec_drop_out: float = None,
    ):
        super().__init__()
        self.num_layers  = int(num_layers)
        self.hidden_size = int(hidden_size)
        self.actual_hidden_dim = self.hidden_size

        _z_drop   = float(drop_out if z_drop_out   is None else z_drop_out)
        _rec_drop = float(drop_out if rec_drop_out  is None else rec_drop_out)

        self.layers = nn.ModuleList([
            TemporalLSTM_V7_NS(
                num_of_nodes = num_of_nodes,
                in_channels  = input_size if i == 0 else self.actual_hidden_dim,
                out_channels = self.hidden_size,
                drop_out     = float(drop_out),
                z_drop_out   = _z_drop,
                rec_drop_out = _rec_drop,
            )
            for i in range(self.num_layers)
        ])

        self.fc = nn.Linear(self.actual_hidden_dim, output_size)
        self._h = None
        self._c = None

    def reset_state(self):
        self._h = None
        self._c = None

    def detach_state_(self):
        if self._h is not None:
            self._h = [t.detach() if t is not None else None for t in self._h]
        if self._c is not None:
            self._c = [t.detach() if t is not None else None for t in self._c]

    def forward(self, x, detach_state=False):
        if (self._h is None) or (self._c is None):
            self._h = [None] * self.num_layers
            self._c = [None] * self.num_layers

        out = x
        h_next, c_next = [], []

        for i, layer in enumerate(self.layers):
            out, h_i, c_i = layer(out, H=self._h[i], C=self._c[i])

            if detach_state:
                h_i, c_i = h_i.detach(), c_i.detach()

            h_next.append(h_i)
            c_next.append(c_i)

        self._h, self._c = h_next, c_next
        return self.fc(out)


# ============================================================
# HELPER – compute L1/L2 penalty for the training loop
# ============================================================
def compute_l2_penalty(model: nn.Module, l2_lambda: float = 1e-4) -> torch.Tensor:
    _LSTM_WEIGHT_KEYS = {"W_i", "W_f", "W_c", "W_o", "U_i", "U_f", "U_c", "U_o"}
    penalty = torch.tensor(0.0, device=next(model.parameters()).device)
    for name, param in model.named_parameters():
        bare = name.split(".")[-1]
        if (
            "gatebias_net" in name
            or "level_net"  in name
            or bare in _LSTM_WEIGHT_KEYS
        ):
            penalty = penalty + param.pow(2).sum()   # L2: w²
    return l2_lambda * penalty


# ============================================================
# TRAINING LOOP WITH NOISE INJECTION
# ============================================================
results = []
df2 = []
start_time = countdown.time()
all_fold = range(k_fold_num)

noise_level = 0.05

for hidden_size in hidden_size_list:

            for num_layers in num_layers_list:
                for drop_out in drop_out_list:

                    num_station = len(cluster_station)

                    for fold_index in all_fold:

                        train_set_homo = []
                        for i in range(len(fold_tensor[fold_index][0])):
                            data_homo = fold_tensor[fold_index][0][i].to_homogeneous()
                            data_homo.x = torch.tensor(extract_x(data_homo), dtype=torch.float32)
                            data_homo.y = data_homo.y[len(data_homo.y)-len(cluster_station):len(data_homo.y)]
                            data_homo.x = data_homo.x.reshape(data_homo.x.shape[0], data_homo.x.shape[2])
                            data_homo.y = data_homo.y.reshape(data_homo.y.shape[0], data_homo.y.shape[2])
                            data_homo.node_type = torch.tensor(np.array(range(num_feature+len(cluster_station))))
                            train_set_homo.append(data_homo)

                        test_set_homo = []
                        for i in range(len(fold_tensor[fold_index][1])):
                            data_homo = fold_tensor[fold_index][1][i].to_homogeneous()
                            data_homo.x = torch.tensor(extract_x(data_homo), dtype=torch.float32)
                            data_homo.node_type = torch.tensor(np.array(range(num_feature+len(cluster_station))))
                            data_homo.y = data_homo.y[len(data_homo.y)-len(cluster_station):len(data_homo.y)]
                            data_homo.x = data_homo.x.reshape(data_homo.x.shape[0], data_homo.x.shape[2])
                            data_homo.y = data_homo.y.reshape(data_homo.y.shape[0], data_homo.y.shape[2])
                            test_set_homo.append(data_homo)

                        val_set_homo = []
                        for i in range(len(fold_tensor[fold_index][2])):
                            data_homo = fold_tensor[fold_index][2][i].to_homogeneous()
                            data_homo.x = torch.tensor(extract_x(data_homo), dtype=torch.float32)
                            data_homo.node_type = torch.tensor(np.array(range(num_feature+len(cluster_station))))
                            data_homo.y = data_homo.y[len(data_homo.y)-len(cluster_station):len(data_homo.y)]
                            data_homo.x = data_homo.x.reshape(data_homo.x.shape[0], data_homo.x.shape[2])
                            data_homo.y = data_homo.y.reshape(data_homo.y.shape[0], data_homo.y.shape[2])
                            val_set_homo.append(data_homo)

                        print(f'running on cluster=_{cluster}, fold={fold_index}, '
                              f'hidden={hidden_size}, layers={num_layers}, dropout={drop_out}')

                        save_path = (
                            f'/content/drive/MyDrive/Workshop Aj.Tohn/pre_trained_models_v23/model_{algo}_'
                            f'Fold={fold_index}_numfeatures={num_feature}_window={window_size}_'
                            f'cluster={cluster}_hidden={hidden_size}_layers={num_layers}_'
                            f'dropout={drop_out}_TBPTT_K={TBPTT_K}'
                        )

                        # Create the directory if it doesn't exist
                        os.makedirs(os.path.dirname(save_path), exist_ok=True)

                        early_stopping = EarlyStopping(
                            patience=patience,
                            min_delta=1e-4,
                            mode='min',
                            save_path=save_path,
                            min_epochs=min_epochs,
                            verbose=False
                        )

                        seed = 0
                        torch.manual_seed(seed)

                        model = PureLSTMModel(
                            num_of_nodes=len(cluster_station) + num_feature,
                            input_size=window_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            drop_out=drop_out,
                            output_size=horizon,
                        ).to(device)

                        optimizer = torch.optim.AdamW(
                            model.parameters(),
                            lr=learn_rate,
                            weight_decay=weight_decay
                        )



                        # ==================== TRAINING (TBPTT) ====================
                        for epoch in tqdm(range(num_epoch)):
                            model.train()
                            model.reset_state()

                            total_loss_value = 0.0
                            total_acc_value = 0.0
                            train_ss_res = 0.0
                            train_ss_tot = 0.0
                            steps = 0
                            chunk_losses = []
                            optimizer.zero_grad(set_to_none=True)

                            for t, snapshot in enumerate(train_set_homo):
                                snapshot = snapshot.to(device)
                                input_x = snapshot.x + torch.randn_like(snapshot.x) * noise_level

                                # ✅ Edge data no longer passed
                                y_hat = model(input_x, detach_state=False)
                                pred  = y_hat[-len(cluster_station):, :]

                                loss = torch.nn.functional.smooth_l1_loss(pred, snapshot.y, beta=1.0)
                                loss = loss + compute_l2_penalty(model, l2_lambda=1e-4)

                                chunk_losses.append(loss)

                                total_loss_value += float(loss.detach().item())
                                total_acc_value  += float(np.mean(accuracy2D_classic(pred.detach(), snapshot.y.detach())) * 100)
                                train_ss_res += torch.sum((snapshot.y - pred.detach()) ** 2).item()
                                train_ss_tot += torch.sum((snapshot.y - torch.mean(snapshot.y)) ** 2).item()
                                steps += 1

                                if (t + 1) % TBPTT_K == 0:
                                    chunk_loss = torch.stack(chunk_losses).mean()
                                    chunk_loss.backward()
                                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                                    optimizer.step()
                                    optimizer.zero_grad(set_to_none=True)
                                    model.detach_state_()
                                    chunk_losses = []

                            if chunk_losses:
                                chunk_loss = torch.stack(chunk_losses).mean()
                                chunk_loss.backward()
                                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                                optimizer.step()
                                optimizer.zero_grad(set_to_none=True)
                                model.detach_state_()
                                chunk_losses = []

                            cost_train = total_loss_value / max(steps, 1)
                            acc_train = total_acc_value / max(steps, 1)
                            r2_train = 1.0 - (train_ss_res / max(train_ss_tot, 1e-8))

                            # ==================== VALIDATION ====================
                            model.eval()
                            model.reset_state()
                            total_cost_val = 0.0
                            total_acc_val = 0.0
                            val_ss_res = 0.0
                            val_ss_tot = 0.0
                            vsteps = 0

                            with torch.no_grad():
                                for vt, snapshot in enumerate(val_set_homo):
                                    snapshot = snapshot.to(device)

                                    # ✅ Edge data no longer passed
                                    y_hat = model(snapshot.x, detach_state=True)
                                    pred = y_hat[-len(cluster_station):, :]

                                    total_cost_val += torch.nn.functional.smooth_l1_loss(pred, snapshot.y, beta=1.0).item()
                                    total_acc_val += float(np.mean(accuracy2D_classic(pred, snapshot.y)) * 100)

                                    val_ss_res += torch.sum((snapshot.y - pred) ** 2).item()
                                    val_ss_tot += torch.sum((snapshot.y - torch.mean(snapshot.y)) ** 2).item()
                                    vsteps += 1

                            cost_val = total_cost_val / max(vsteps, 1)
                            acc_val = total_acc_val / max(vsteps, 1)
                            r2_val = 1.0 - (val_ss_res / max(val_ss_tot, 1e-8))

                            if (epoch + 1) % LOG_EVERY == 0:
                                print(f"Epoch {epoch+1:03d}/{num_epoch} | "
                                      f"Train Loss: {cost_train:.4f}  Acc: {acc_train:.2f}%  R2: {r2_train:.4f} | "
                                      f"Val Huber Loss: {cost_val:.4f} Acc: {acc_val:.2f}% R2: {r2_val:.4f}")

                            if early_stopping(cost_val, cost_train, model, epoch):
                                print(f"AVG train Accuracy: {acc_train:.4f}")
                                print(f"AVG train Loss: {cost_train:.4f}")
                                print(f"AVG train R2: {r2_train:.4f}")
                                print(f"AVG val Accuracy: {acc_val:.4f}")
                                print(f"AVG val Loss: {cost_val:.4f}")
                                print(f"AVG val R2: {r2_val:.4f}")
                                print("Early stopping triggered")
                                break

                        early_stopping.plot_loss_history()

                        if save_path is not None:
                            checkpoint = torch.load(save_path + '.pth', map_location=device)
                            model.load_state_dict(checkpoint['model_state_dict'])

                        # ==================== TEST ====================
                        model.eval()
                        model.reset_state()
                        cost, acc, time_acc, rmse = 0.0, 0.0, 0.0, 0.0
                        test_ss_res, test_ss_tot, nse_score_test, tsteps = 0.0, 0.0, 0.0, 0

                        with torch.no_grad():
                            for tt, snapshot in enumerate(test_set_homo):
                                snapshot = snapshot.to(device)

                                # ✅ Edge data no longer passed
                                y_hat = model(snapshot.x, detach_state=True)
                                predictions = y_hat[-len(cluster_station):, :]
                                observations = snapshot.y

                                mse_t = torch.mean((predictions - observations) ** 2)
                                cost += float(mse_t.item())
                                rmse += float(torch.sqrt(mse_t).item())
                                acc += float(np.mean(accuracy2D_classic(predictions, observations)) * 100)
                                time_acc += np.mean(
                                    accuracy2D_classic(predictions, observations).reshape(observations.shape[0], observations.shape[1]),
                                    axis=0
                                )
                                nse_score_test += float(np.mean(np.array(list(nse_per_station(observations, predictions).values()))))

                                test_ss_res += torch.sum((observations - predictions) ** 2).item()
                                test_ss_tot += torch.sum((observations - torch.mean(observations)) ** 2).item()
                                tsteps += 1

                        acc_test = acc / max(tsteps, 1)
                        mse_test = cost / max(tsteps, 1)
                        rmse_test = rmse / max(tsteps, 1)
                        nse_score_test = nse_score_test / max(tsteps, 1)
                        r2_test = 1.0 - (test_ss_res / max(test_ss_tot, 1e-8))

                        print(f"AVG test Accuracy: {acc_test:.4f}")
                        print(f"AVG test MSE: {mse_test:.4f}")
                        print(f"AVG test RMSE: {rmse_test:.4f}")
                        print(f"AVG test R2: {r2_test:.4f}")
                        print(f"AVG Nash-Sutcliffe Efficiency (NSE): {nse_score_test:.4f}")

                        row = np.hstack((
                            np.array([fold_index, num_feature, window_size, cluster, checkpoint['epoch'],
                                       hidden_size, num_layers, drop_out,
                                      acc_train, cost_train, r2_train,
                                      acc_test, mse_test, rmse_test, r2_test,
                                      nse_score_test, checkpoint['validation_metric'], r2_val], dtype=float),
                            np.array(time_acc, dtype=float).ravel()
                        ))
                        df2.append(row)
                        df = pd.DataFrame(df2)

                        df.columns = [
                            'fold_index','num_feature','window_size','cluster','epoch','hidden_size',
                            'num_layers','drop_out','acc_train','adaptive_huber_loss_train','r2_train',
                            'acc_test','mse_test','rmse_test','r2_test','nse_score_test','adaptive_huber_loss_val','r2_val',
                            'acc_month1','acc_month2','acc_month3','acc_month4','acc_month5','acc_month6',
                            'acc_month7','acc_month8','acc_month9','acc_month10','acc_month11','acc_month12'
                        ]
                        df.to_csv(
                            f'/content/drive/MyDrive/Workshop Aj.Tohn/metric_results_v23/predict_{algo}_cluster={cluster}_'
                            f'numfeatures={num_feature}_window={window_size}_TBPTT_K={TBPTT_K}.csv',
                            index=False
                        )

end_time = countdown.time()
elapsed_time = end_time - start_time
results.append({'cluster': cluster, 'time': elapsed_time})
results_df = pd.DataFrame(results)
results_df.to_csv(f'/content/drive/MyDrive/Workshop Aj.Tohn/metric_results_v23/predict_time_{algo}_cluster={cluster}.csv', index=False)

running on cluster=_7, fold=0, hidden=128, layers=2, dropout=0.4


  7%|▋         | 20/300 [01:54<26:16,  5.63s/it]

Epoch 020/300 | Train Loss: 0.2639  Acc: 52.30%  R2: 0.6282 | Val Huber Loss: 0.2110 Acc: 42.72% R2: 0.5132


 13%|█▎        | 40/300 [03:48<25:28,  5.88s/it]

Epoch 040/300 | Train Loss: 0.1990  Acc: 53.08%  R2: 0.6438 | Val Huber Loss: 0.1894 Acc: 45.43% R2: 0.5685


...

 47%|████▋     | 140/300 [13:05<14:55,  5.59s/it]

Epoch 140/300 | Train Loss: 0.1556  Acc: 54.25%  R2: 0.6663 | Val Huber Loss: 0.1603 Acc: 49.15% R2: 0.6399


 53%|█████▎    | 160/300 [14:57<13:04,  5.60s/it]

Epoch 160/300 | Train Loss: 0.1568  Acc: 54.16%  R2: 0.6654 | Val Huber Loss: 0.1543 Acc: 50.11% R2: 0.6567


 57%|█████▋    | 172/300 [16:07<12:36,  5.91s/it]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

for station_idx in range(len(cluster_station)):
    if station_idx <10:

        for time, snapshot in enumerate(train_set_homo):
            #if time % 12 == 0:
                snapshot = snapshot.to(device)

                x = snapshot.x.detach().cpu().numpy()   # [num_nodes, window]
                y = snapshot.y.detach().cpu().numpy()   # [num_stations, horizon]

                # rainfall
                x_rain = x[num_feature + station_idx]   # [window]
                y_rain = y[station_idx]                 # [horizon]

                # climate indices
                x_climate = x[:num_feature]             # [num_feature, window]

                climate_names = list(feature_climate[selected_index])
                window_size = x_rain.shape[0]
                horizon = y_rain.shape[0]

                t_window = np.arange(1, window_size + 1)
                t_horizon = np.arange(window_size + 1, window_size + horizon + 1)

                fig, axes = plt.subplots(
                    2, 1, figsize=(14, 8), sharex=True,
                    gridspec_kw={"height_ratios": [2, 1]}
                )

                # =========================
                # (A) Rainfall
                # =========================
                axes[0].plot(t_window, x_rain, label=f"Observed Rainfall (input) ({window_size})", linewidth=2)
                axes[0].plot(t_horizon, y_rain, label=f"Observed Rainfall (target) ({horizon})", linestyle="--", linewidth=2)

                axes[0].axvline(window_size, color="gray", linestyle=":")
                axes[0].set_ylabel("Rainfall (normalise)")
                axes[0].set_title(
                    f"Attention_LSTM – Station {node['Node'].iloc[cluster_station[station_idx]]} | snapshot t={time}"
                )
                axes[0].grid(alpha=0.3)

                model.eval()
                with torch.no_grad():
                    y_pred_all = model(snapshot.x)  # [num_nodes, horizon]
                    y_pred_station = y_pred_all[-len(cluster_station):, :][station_idx].detach().cpu().numpy()  # [horizon]

                # ✅ FIX: plot y_pred_station directly (no extra [station_idx])
                axes[0].plot(t_horizon, y_pred_station, label="Rainfall prediction (2024)", linewidth=2)
                axes[0].legend()

                # =========================
                # (B) Climate indices (named)
                # =========================
                for i, name in enumerate(climate_names):
                    axes[1].plot(t_window, x_climate[i], linewidth=1.8, alpha=0.85, label=name)

                axes[1].axvline(window_size, color="gray", linestyle=":")
                axes[1].set_ylabel("Climate index")
                axes[1].set_xlabel("Month")

                axes[1].legend(
                    loc="upper center",
                    bbox_to_anchor=(0.5, -0.25),
                    ncol=min(4, len(climate_names)),
                    fontsize=9,
                    frameon=False
                )
                axes[1].grid(alpha=0.3)

                plt.tight_layout()
                plt.show()
